# Purpose:
# 1. Load the final SQL outputs.
# 2. Profile and validate the customer decision table.
# 3. Independently reproduce SQL results in Python.
# 4. Standardize fields and handle known data limitations.
# 5. Merge customer results with segment-level context using validated joins.
# 6. Review the main analytical findings.
# 7. Export clean customer- and segment-level files for Tableau.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Run from the repository root or from this notebook's folder.
PROJECT_DIR = Path.cwd()
if (PROJECT_DIR / "Python_Group_11_Capstone").is_dir():
    PROJECT_DIR = PROJECT_DIR / "Python_Group_11_Capstone"
DATA_DIR = PROJECT_DIR / "Csv_Files"
OUTPUT_DIR = PROJECT_DIR / "Output_Tableau_CSVs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Input folder:", DATA_DIR)
print("Output folder:", OUTPUT_DIR)


In [ ]:

CUSTOMER_COLS = [
    "user_id",
    "basket_segment",
    "frequency_segment",
    "total_prior_orders",
    "prior_gap_count",
    "avg_basket_size",
    "avg_days_between_orders",
    "reorder_rate_pct",
    "latest_observed_gap",
    "expected_order_gap",
    "gap_vs_expected_ratio",
    "engagement_status",
    "latest_gap_capped_flag",
    "insufficient_history_flag",
    "assessed_flag",
    "at_risk_flag",
]

SEGMENT_COLS = [
    "basket_segment",
    "frequency_segment",
    "segment_label",
    "segment_customer_count",
    "pct_of_customer_base",
    "segment_pct_at_risk_among_assessed",
]

VALIDATION_COLS = [
    "user_id",
    "prior_orders",
    "total_items",
    "valid_gap_count",
    "total_gap_days",
    "reordered_items",
]

REFERENCE_COLS = [
    "metric",
    "sql_value",
]

customers = pd.read_csv(
    DATA_DIR / "submission_customer_decision.csv",
    header=None,
    names=CUSTOMER_COLS
)

segments = pd.read_csv(
    DATA_DIR / "submission_segment_context.csv",
    header=None,
    names=SEGMENT_COLS
)

validation = pd.read_csv(
    DATA_DIR / "python_validation_input.csv",
    header=None,
    names=VALIDATION_COLS
)

reference = pd.read_csv(
    DATA_DIR / "sql_validation_reference.csv",
    header=None,
    names=REFERENCE_COLS
)

print("Customers:", customers.shape)
print("Segments:", segments.shape)
print("Validation:", validation.shape)
print("Reference:", reference.shape)

print("\nCustomer columns:")
print(customers.columns.tolist())

In [ ]:
# Load the four SQL exports



In [ ]:
# 2. Profile the customer decision table

# Summarize
profile = pd.Series({
    "rows": len(customers),
    "columns": customers.shape[1],
    "unique_user_id": customers["user_id"].nunique(),
    "duplicate_user_id": customers["user_id"].duplicated().sum(),
    "missing_values": customers.isna().sum().sum(),
})

# Review each column's data type and missing values
column_check = pd.DataFrame({
    "data_type": customers.dtypes.astype(str),
    "missing_values": customers.isna().sum(),
})

display(profile.to_frame("value"))
display(column_check)

# Review the main classification fields
for column in ["basket_segment", "frequency_segment", "engagement_status"]:
    display(customers[column].value_counts().to_frame("customers"))

# Confirm one complete row per customer
assert len(customers) == 206_209, "Unexpected customer row count."
assert customers["user_id"].is_unique, "Duplicate customer IDs found."
assert customers.isna().sum().sum() == 0, "Missing source values found."

3. Reconcile Python calculations to SQL

In [ ]:


# Calculating weighted metrics from the totals.
totals = validation.sum(numeric_only=True)

python_metrics = pd.Series({
    "total_prior_orders":
        totals["prior_orders"],

    "weighted_avg_basket_size":
        totals["total_items"] / totals["prior_orders"],

    "avg_days_between_orders":
        totals["total_gap_days"] / totals["valid_gap_count"],

    "overall_reorder_rate_pct":
        100 * totals["reordered_items"] / totals["total_items"],
})


checks = reference.set_index("metric")
checks["python_value"] = python_metrics

checks["pass"] = np.isclose(
    checks["sql_value"],
    checks["python_value"],
)

display(checks.round(2))

assert checks["pass"].all(), "Python results do not match SQL."

 4. Standardize fields and handle analytical limitations


In [ ]:
# 4. Standardizing fields and handle analytical limitations

#  1: Standardize numeric data types.
integer_cols = [
    "user_id",
    "total_prior_orders",
    "prior_gap_count",
    "latest_observed_gap",
    "latest_gap_capped_flag",
    "insufficient_history_flag",
    "assessed_flag",
    "at_risk_flag",
]

numeric_cols = [
    "avg_basket_size",
    "avg_days_between_orders",
    "reorder_rate_pct",
    "expected_order_gap",
    "gap_vs_expected_ratio",
]

customers[integer_cols] = customers[integer_cols].astype(int)
customers[numeric_cols] = customers[numeric_cols].apply(pd.to_numeric)


# 2: Standardize categorical labels.
category_cols = [
    "basket_segment",
    "frequency_segment",
    "engagement_status",
]

customers[category_cols] = customers[category_cols].apply(
    lambda column: column.str.strip()
)

assert set(customers["basket_segment"]) == {
    "Small", "Medium", "Large"
}

assert set(customers["frequency_segment"]) == {
    "Frequent", "Regular", "Infrequent"
}


# 3: A recorded 30-day latest gap may represent
# 30 or more days, so do not use its exact slowdown ratio.
customers["gap_vs_expected_ratio_clean"] = customers[
    "gap_vs_expected_ratio"
].mask(
    customers["latest_gap_capped_flag"].eq(1)
)


# 4: Keeping customers with insufficient history
# instead of deleting them. Use the SQL assessed_flag
# to identify the precise analytical population.
assert customers["assessed_flag"].isin([0, 1]).all()

print(
    "Capped ratios set to missing:",
    customers["gap_vs_expected_ratio_clean"].isna().sum()
)

print(
    "Precisely assessed customers:",
    customers["assessed_flag"].sum()
)

print(
    "Insufficient-history customers retained:",
    customers["insufficient_history_flag"].sum()
)

# 5. Add segment context with a validated many-to-one merge

In [ ]:

# Merge the 9-row segment context onto the customer-level table.
# validate="many_to_one" prevents an unexpected fan-out.
# indicator=True confirms every customer found a matching segment.
# Customer row count must remain unchanged after the merge.
# Calculating risk using precisely assessed customers only.

segment_risk = (
    customers.loc[customers["assessed_flag"].eq(1)]
    .groupby(
        ["basket_segment", "frequency_segment"],
        observed=True,
    )["at_risk_flag"]
    .mean()
    .mul(100)
    .round(2)
    .rename("python_pct_at_risk_among_assessed")
    .reset_index()
)

# Add Python risk calculation to SQL segment context.
segment_context = segments.merge(
    segment_risk,
    on=["basket_segment", "frequency_segment"],
    validate="one_to_one",
)

# Confirm Python independently matches SQL.
segment_context["risk_validation_pass"] = np.isclose(
    segment_context["segment_pct_at_risk_among_assessed"],
    segment_context["python_pct_at_risk_among_assessed"],
    atol=0.01,
    rtol=0,
)

assert segment_context["risk_validation_pass"].all(), (
    "Python segment-risk results do not match SQL."
)

# Adding segment context to the customer-level table.
merged = customers.merge(
    segment_context,
    on=["basket_segment", "frequency_segment"],
    how="left",
    validate="many_to_one",
    indicator=True,
)

assert len(merged) == len(customers) == 206_209
assert len(segment_context) == 9
assert merged["_merge"].eq("both").all()

segment_summary = (
    segment_context
    .rename(columns={
        "segment_customer_count": "customers",
        "pct_of_customer_base": "pct_base",
        "segment_pct_at_risk_among_assessed": "at_risk_pct",
    })
    [
        [
            "segment_label",
            "customers",
            "pct_base",
            "at_risk_pct",
            "python_pct_at_risk_among_assessed",
            "risk_validation_pass",
        ]
    ]
    .sort_values("at_risk_pct", ascending=False)
    .reset_index(drop=True)
)

display(segment_summary)

print("Rows before merge:", len(customers))
print("Rows after merge:", len(merged))
print("Validated many-to-one merge passed.")

In [ ]:
# 6. Confirming the main analytical findings

# Averaging customer gap by basket segment.
basket_gap = (
    customers.groupby(
        "basket_segment",
        observed=True
    )["avg_days_between_orders"]
    .mean()
    .reindex(["Small", "Medium", "Large"])
    .round(2)
)

# Average customer gap by frequency segment.
frequency_gap = (
    customers.groupby(
        "frequency_segment",
        observed=True
    )["avg_days_between_orders"]
    .mean()
    .reindex(["Frequent", "Regular", "Infrequent"])
    .round(2)
)

# Percentage of customers in each engagement status.
status_mix = (
    customers["engagement_status"]
    .value_counts(normalize=True)
    .mul(100)
    .reindex([
        "Stable",
        "Weakening",
        "High Concern",
        "Capped Gap - Review",
        "Insufficient History",
    ])
    .round(2)
)

display(
    basket_gap.rename("avg_gap_days").to_frame(),
    frequency_gap.rename("avg_gap_days").to_frame(),
    status_mix.rename("pct_customers").to_frame(),
)

In [ ]:
# 7. Export Tableau-ready files

# A. CUSTOMER-LEVEL TABLE
# One row per customer.
# Used for engagement status, customer behavior,
# expected vs. latest cadence, and at-risk analysis.


customer_tableau_cols = [
    "user_id",
    "basket_segment",
    "frequency_segment",
    "total_prior_orders",
    "avg_basket_size",
    "avg_days_between_orders",
    "reorder_rate_pct",
    "latest_observed_gap",
    "expected_order_gap",
    "gap_vs_expected_ratio_clean",
    "engagement_status",
    "assessed_flag",
    "at_risk_flag",
]

customer_tableau = merged[customer_tableau_cols].copy()

# Verify one row per customer.
assert customer_tableau.shape == (206_209, 13)
assert customer_tableau["user_id"].is_unique
assert customer_tableau.columns.duplicated().sum() == 0

customer_output = OUTPUT_DIR / "final_customer_decision_tableau.csv"

customer_tableau.to_csv(
    customer_output,
    index=False,
)



# B. SEGMENT-LEVEL TABLE
# One row per basket x frequency segment.
# Used to compare the 9 behavioral groups.


segment_tableau_cols = [
    "basket_segment",
    "frequency_segment",
    "segment_label",
    "segment_customer_count",
    "pct_of_customer_base",
    "segment_pct_at_risk_among_assessed",
]

segment_tableau = (
    segment_context[segment_tableau_cols]
    .sort_values(
        "segment_pct_at_risk_among_assessed",
        ascending=False,
    )
    .reset_index(drop=True)
)

# Verify exactly 9 unique behavioral segments.
assert segment_tableau.shape == (9, 6)
assert not segment_tableau.duplicated(
    ["basket_segment", "frequency_segment"]
).any()

segment_output = OUTPUT_DIR / "segment_summary_tableau.csv"

segment_tableau.to_csv(
    segment_output,
    index=False,
)



# Final checks


assert customer_output.exists()
assert segment_output.exists()

print("Customer Tableau file:")
print(customer_output)
print("Shape:", customer_tableau.shape)

print("\nSegment Tableau file:")
print(segment_output)
print("Shape:", segment_tableau.shape)

print("\nCustomer columns:")
print(customer_tableau.columns.tolist())

print("\nSegment columns:")
print(segment_tableau.columns.tolist())

## Business Interpretation

### What the cleaned data shows

Of the 206,209 customers in the final customer base, 119,993 have enough
uncapped historical information to be assessed against customers with similar
shopping behavior.

Among those assessed customers, 17,678 are classified as Weakening and
12,034 as High Concern. Together, 29,712 customers, or 24.76% of the
assessed population, are returning more slowly than the historical thresholds
used for their comparison groups.

90,281 customers are classified as Stable.

Average historical order gaps are similar across Small, Medium, and Large
basket groups, suggesting that basket size alone does not explain ordering
cadence.

### Limitations

The dataset records order gaps only up to 30 days. Because a value of 30 may
represent 30 or more days, customers whose latest observed gap reaches this
limit are flagged for review rather than assigned an exact slowdown ratio.

43,576 customers also have insufficient historical order information for the
slowdown assessment. They remain in the final customer base rather than being
deleted.

The latest gap is observed only when another order occurs. Therefore this
analysis identifies historically slower observed returns; it does not show
which customers are currently overdue or prove that a customer will churn.

### Recommended action

Use the engagement status to test targeted customer-stage rewards rather than
sending the same incentive to every customer. Begin with a controlled pilot:
use a smaller time-limited incentive for Weakening customers and a stronger
one-time win-back offer for High Concern customers, while keeping Stable
customers on lighter loyalty or milestone rewards.

Measure the pilot using repeat-order rate, time to next order, basket size,
and movement from Weakening back to Stable.